In [ ]:
import os
import sys
import json
import numpy as np
from urchin import URDF
import trimesh
import pickle
import quaternion
from scipy.spatial.distance import cdist
import pyrender

In [ ]:
import pybullet as p

In [ ]:
sys.path.append("..")

from utils.grasp_utils import get_handmodel, rotation_matrix_from_vectors

from utils.grasp_utils import get_urdf_path, convert_aligned_to_gripper_pose, convert_gripper_to_aligned_pose, get_quat_np, get_quat_pyb, get_base_pose


In [ ]:
p.connect(p.GUI) # with GUI, useful to see final result
# p.connect(p.DIRECT) # without GUI, useful when coding and trying out pybullet functions

In [ ]:
urdf_dir = "../grippers/"

In [ ]:
source_gripper = "mano_right"
target_gripper = "fetch_gripper"
device = "cpu"

In [ ]:
source_model = get_handmodel(
  source_gripper,
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)

sm_left = get_handmodel(
  "mano_left",
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)

target_model = get_handmodel(
  target_gripper,
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)

In [ ]:
src_pose_xyzw = convert_aligned_to_gripper_pose([0, 0, 0, 0, 0, 0, 1], source_gripper)
src_pose_wxyz = [*src_pose_xyzw[:3], src_pose_xyzw[-1], *src_pose_xyzw[3:-1]]
src_pose_tf = get_base_pose(src_pose_wxyz)

src_urdf_filename = os.path.join(urdf_dir, get_urdf_path(source_gripper))


In [ ]:
default_pose_xyzw = [0, 0, 0, 0, 0, 0, 1]
target_pose_xyzw = convert_aligned_to_gripper_pose(default_pose_xyzw, target_gripper)
target_pose_wxyz = [*target_pose_xyzw[:3], target_pose_xyzw[-1], *target_pose_xyzw[3:-1]]
target_pose_tf = get_base_pose(target_pose_wxyz)

target_urdf_filename = os.path.join(urdf_dir, get_urdf_path(target_gripper))

In [ ]:
id_main_gripper = p.loadURDF(src_urdf_filename)
id_other_gripper = p.loadURDF(target_urdf_filename)

In [ ]:
p.resetBasePositionAndOrientation(id_main_gripper, src_pose_xyzw[:3], src_pose_xyzw[3:])

In [ ]:
p.resetBasePositionAndOrientation(id_other_gripper, target_pose_xyzw[:3], target_pose_xyzw[3:])